# 08 — EDA Case Studies
## End-to-end Exploratory Data Analysis

This notebook walks through three synthetic scenarios: retail sales performance, student performance, and customer churn. Each case follows the same workflow: business question → data quality → descriptive analysis → visualization → interpretation → next steps.

**Important:** all records are synthetic examples created for learning. Do not present their patterns as real-world findings.


## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme()
pd.set_option("display.max_columns", None)


## Case Study 1 — Retail Sales Performance
**Business question:** Which regions and categories contribute most to sales, and what data-quality issues need attention?

### 1.1 Create the dataset

In [ ]:
rng = np.random.default_rng(2026)
n_retail = 160
retail_df = pd.DataFrame({
    "order_id": [f"ORD-{i:04d}" for i in range(1, n_retail + 1)],
    "region": rng.choice(["North", "South", "East", "West"], n_retail),
    "category": rng.choice(["Electronics", "Home", "Clothing", "Books"], n_retail),
    "quantity": rng.integers(1, 8, n_retail),
    "unit_price": rng.choice([10.0, 19.99, 29.5, 49.0, 89.0, 149.0], n_retail),
    "discount": rng.choice([0.0, 0.05, 0.10, 0.15], n_retail),
    "order_date": pd.date_range("2025-01-01", periods=n_retail, freq="2D")
})
retail_df["sales"] = (retail_df["quantity"] * retail_df["unit_price"] * (1 - retail_df["discount"])).round(2)
retail_df.loc[7, "sales"] = 9999.00  # deliberately unusual value for investigation
retail_df.loc[rng.choice(retail_df.index, 5, replace=False), "discount"] = np.nan
retail_df = pd.concat([retail_df, retail_df.iloc[[3]]], ignore_index=True)

retail_df.head()


### 1.2 Inspect data quality

In [ ]:
print("Shape:", retail_df.shape)
display(retail_df.dtypes.rename("dtype").to_frame())
display(pd.DataFrame({
    "missing_count": retail_df.isna().sum(),
    "missing_percent": retail_df.isna().mean().mul(100).round(2)
}))
print("Exact duplicate rows:", retail_df.duplicated().sum())
display(retail_df.describe(include="all").T)


### 1.3 Sales by region and category

In [ ]:
region_summary = retail_df.groupby("region")["sales"].agg(
    order_count="count", total_sales="sum", average_order_value="mean", median_order_value="median"
).sort_values("total_sales", ascending=False)
category_summary = retail_df.groupby("category")["sales"].agg(
    order_count="count", total_sales="sum", average_order_value="mean", median_order_value="median"
).sort_values("total_sales", ascending=False)

display(region_summary.round(2))
display(category_summary.round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
region_summary["total_sales"].plot(kind="bar", ax=axes[0], title="Total Sales by Region")
axes[0].set_xlabel("Region")
axes[0].set_ylabel("Sales")
category_summary["total_sales"].plot(kind="bar", ax=axes[1], title="Total Sales by Category")
axes[1].set_xlabel("Category")
axes[1].set_ylabel("Sales")
axes[1].tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()


### 1.4 Distribution and unusual values

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=retail_df, x="sales", bins=15, kde=True, ax=axes[0])
axes[0].set_title("Retail Sales Distribution")
sns.boxplot(data=retail_df, x="sales", ax=axes[1])
axes[1].set_title("Potential Outliers in Sales")
plt.tight_layout()
plt.show()

display(retail_df.sort_values("sales", ascending=False).head(8))


### 1.5 Case 1 findings template
Use the outputs above to name the top region/category, cite counts and values, and discuss the unusual transaction. The dataset is synthetic; validate any proposed business action with real data.

## Case Study 2 — Student Performance
**Business question:** How are scores distributed, and are study hours or attendance associated with exam scores in this sample?

### 2.1 Create the dataset

In [ ]:
rng = np.random.default_rng(73)
n_students = 150
departments = rng.choice(["Computer Science", "Business", "Engineering", "Arts"],
                         n_students, p=[0.35, 0.22, 0.28, 0.15])
study_hours = np.clip(rng.gamma(3.0, 2.5, n_students), 0.5, 24)
attendance = np.clip(rng.normal(82, 10, n_students), 40, 100)
dept_effect = pd.Series(departments).map({
    "Computer Science": 3, "Business": 0, "Engineering": 2, "Arts": -2
}).to_numpy()
scores = np.clip(42 + 1.2 * study_hours + 0.2 * (attendance - 70)
                 + dept_effect + rng.normal(0, 11, n_students), 0, 100)

student_df = pd.DataFrame({
    "student_id": [f"STU-{i:03d}" for i in range(1, n_students + 1)],
    "study_hours": study_hours.round(1),
    "attendance_pct": attendance.round(1),
    "exam_score": scores.round(1),
    "department": departments,
    "study_mode": rng.choice(["Self-study", "Online course", "Classroom", "Hybrid"], n_students)
})
student_df.loc[rng.choice(student_df.index, 6, replace=False), "exam_score"] = np.nan
student_df.head()


### 2.2 Inspect and summarize

In [ ]:
display(student_df.isna().sum().to_frame("missing_count"))
display(student_df[["study_hours", "attendance_pct", "exam_score"]].describe().T)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=student_df, x="exam_score", bins=15, kde=True, ax=axes[0])
axes[0].set_title("Exam Score Distribution")
sns.histplot(data=student_df, x="study_hours", bins=15, kde=True, ax=axes[1])
axes[1].set_title("Weekly Study Hours")
plt.tight_layout()
plt.show()


### 2.3 Compare scores and associations

In [ ]:
department_scores = student_df.groupby("department")["exam_score"].agg(
    count="count", mean="mean", median="median", std="std"
).sort_values("median", ascending=False)
display(department_scores.round(2))

sns.boxplot(data=student_df, x="department", y="exam_score")
plt.title("Exam Scores by Department")
plt.xlabel("Department")
plt.ylabel("Exam score")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

pair = student_df[["study_hours", "exam_score"]].dropna()
print("Pairwise complete rows:", len(pair))
print("Pearson correlation:", pair["study_hours"].corr(pair["exam_score"], method="pearson"))
print("Spearman correlation:", pair["study_hours"].corr(pair["exam_score"], method="spearman"))

sns.regplot(data=student_df, x="attendance_pct", y="exam_score",
            scatter_kws={"alpha": 0.65})
plt.title("Attendance vs Exam Score")
plt.xlabel("Attendance (%)")
plt.ylabel("Exam score")
plt.tight_layout()
plt.show()


### 2.4 Case 2 interpretation
Correlation and group differences are descriptive. Prior knowledge, course difficulty, motivation, resources, and other factors may affect the patterns. These synthetic data cannot establish causation.

## Case Study 3 — Customer Churn Exploration
**Business question:** How do tenure and monthly charges differ between customers who churned and customers who stayed?

### 3.1 Create the dataset

In [ ]:
rng = np.random.default_rng(91)
n_customers = 220
tenure = rng.integers(1, 73, n_customers)
monthly_charges = np.clip(rng.normal(68, 18, n_customers), 20, 130).round(2)
contract_type = rng.choice(["Month-to-month", "One year", "Two year"], n_customers,
                           p=[0.55, 0.25, 0.20])
# Synthetic probability rule solely to create a varied practice label.
logit = 1.0 - 0.035 * tenure + 0.012 * (monthly_charges - 65)
logit += np.where(contract_type == "Month-to-month", 0.8, 0)
prob_churn = 1 / (1 + np.exp(-logit))
churn = np.where(rng.random(n_customers) < prob_churn, "Yes", "No")

churn_df = pd.DataFrame({
    "customer_id": [f"CUS-{i:04d}" for i in range(1, n_customers + 1)],
    "tenure_months": tenure,
    "monthly_charges": monthly_charges,
    "contract_type": contract_type,
    "churn": churn,
    "support_tickets": rng.poisson(1.8, n_customers)
})
churn_df.loc[rng.choice(churn_df.index, 5, replace=False), "monthly_charges"] = np.nan
churn_df.head()


### 3.2 Churn rate and group comparisons

In [ ]:
churn_counts = churn_df["churn"].value_counts()
churn_percent = churn_df["churn"].value_counts(normalize=True).mul(100).round(2)
display(pd.DataFrame({"count": churn_counts, "percent": churn_percent}))

churn_summary = churn_df.groupby("churn")[["tenure_months", "monthly_charges", "support_tickets"]].agg(
    ["count", "mean", "median", "std"]
)
display(churn_summary.round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=churn_df, x="churn", y="tenure_months", ax=axes[0])
axes[0].set_title("Tenure by Churn Status")
sns.boxplot(data=churn_df, x="churn", y="monthly_charges", ax=axes[1])
axes[1].set_title("Monthly Charges by Churn Status")
plt.tight_layout()
plt.show()


### 3.3 Churn by contract type

In [ ]:
contract_counts = pd.crosstab(churn_df["contract_type"], churn_df["churn"])
contract_percent = pd.crosstab(
    churn_df["contract_type"], churn_df["churn"], normalize="index"
).mul(100).round(2)
display(contract_counts)
display(contract_percent)

contract_counts.plot(kind="bar", stacked=True)
plt.title("Churn Counts by Contract Type")
plt.xlabel("Contract type")
plt.ylabel("Customers")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


### 3.4 Case 3 interpretation
Report counts and denominators alongside churn percentages. Observed group differences do not prove a variable causes churn. Real decisions require validation on real customer data and additional context.

## Final report template

For each case study, document:

1. **Objective:** What question did you investigate?
2. **Data quality:** What was missing, duplicated, invalid, or unusual?
3. **Methods:** Which statistics and visualizations did you choose, and why?
4. **Findings:** Three to five observations supported by numbers or charts.
5. **Recommendations:** What should be checked or done next?
6. **Limitations:** What cannot be concluded from these data?

Continue with `practice.md`, then compare your analysis with `solution.md`.
